# 🎢 Carro-péndulo: aprendizaje por refuerzo frente a control clásico (PID, LQR y DQN)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/07_reinforcement_learning/n19_cartpole_rl_vs_control.ipynb)

> **Tiempo:** unos 5–8 min en CPU (Colab o portátil). No necesita GPU ni `gym`: la física está escrita en NumPy en este mismo notebook.
> Si quieres probar el código deprisa, pon `QUICK_RUN = True` (entrena menos y las cifras serán peores).

En n15 aprendimos Q-learning con una tabla y en n16 DQN con el Lunar Lander. Aquí volvemos al **problema clásico de control**: mantener
un péndulo invertido sobre un carro que se mueve por un raíl. Es el mismo problema con el que Barto, Sutton y Anderson (1983) presentaron el
aprendizaje por refuerzo, y el ejemplo de libro del control moderno. Por eso sirve para **comparar las dos culturas con la misma planta**:

| | Control clásico / moderno | Aprendizaje por refuerzo |
|---|---|---|
| Qué necesitas | un **modelo** de la planta (ecuaciones) | un **simulador** o la planta real y muchas pruebas |
| Qué diseñas | ganancias (PID) o pesos $Q, R$ (LQR) | la **recompensa**, las acciones y la red |
| Qué obtienes | una ley $u = -Kx$ con garantías cerca del equilibrio | una política $\pi(s)$ aprendida de la experiencia |
| Coste / objetivo | minimizar $\sum x^\top Q x + u^\top R u$ | maximizar $\sum \gamma^k r_k$ |

**🏭 ¿Por qué importa en la industria?** La misma dinámica aparece en un **puente grúa** (la carga colgante es un péndulo que hay que llevar
de un sitio a otro sin que oscile), en robots móviles que se auto-equilibran, en brazos con cargas flexibles o en el aterrizaje de un cohete (n16).
En planta casi siempre hay un PID o un LQR funcionando; la pregunta práctica es **cuándo merece la pena aprender un controlador con RL**
y qué riesgos tiene (datos, robustez, seguridad).

**Qué vas a hacer**
1. Simular la física del carro-péndulo y **verla** en animaciones.
2. Estabilizarlo con un **PID** (dos lazos) y con un **LQR**, y ajustar ganancias con *sliders*.
3. Entrenar un agente **DQN** en PyTorch y ver cómo se comporta **antes, durante y después** del entrenamiento.
4. Mirar **qué ha aprendido** (mapa de la política frente a la ley del LQR).
5. Comparar RL y LQR ante **empujones, cambios de masa y ruido de medida**.
6. Aprender el **modelo** en lugar de la política (identificación + LQR) y comparar cuántos datos necesita cada enfoque.

Los bloques **✍️ Ejercicio** son para vosotros; el cuestionario `n19_cartpole_rl_vs_control_quiz.md` pregunta por **vuestros** resultados.

In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
# 📦 Librerías y configuración
import os
import copy
import time
import random
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle, FancyArrow
from IPython.display import HTML, display
from scipy.linalg import expm, solve_discrete_are
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore', message='Animation was deleted')

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
HEADLESS = os.environ.get('HEADLESS', '0') == '1'      # ejecución sin pantalla (p. ej. nbconvert): sin widgets
USE_WIDGETS = HAS_WIDGETS and not HEADLESS

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'    # True = prueba rápida (entrena menos)
TOTAL_STEPS = 30_000 if QUICK_RUN else 80_000          # pasos de interacción del agente DQN (≈ 27 min de "planta real" a 50 Hz)
ANIM_MODE = os.environ.get('ANIM_MODE', 'jshtml')      # 'jshtml' (reproductor en el notebook), 'static' (tira de fotogramas) o 'none'
SAVE_GIFS = True                                       # guarda las animaciones principales como GIF en FIG_DIR
FIG_DIR = Path('./figures_cartpole'); FIG_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)    # una red tan pequeña va más rápida en CPU que en GPU (el cuello de botella es el simulador)
print(f'PyTorch {torch.__version__} | widgets: {USE_WIDGETS} | QUICK_RUN: {QUICK_RUN} | pasos DQN: {TOTAL_STEPS:,}')

## 1. La planta: física del carro-péndulo

Estado $s = [x,\ \dot x,\ \theta,\ \dot\theta]$: posición y velocidad del carro, ángulo del péndulo ($\theta = 0$ es **vertical hacia arriba**,
$\theta>0$ inclina la punta hacia $+x$) y su velocidad angular. La entrada es la fuerza $u$ sobre el carro, limitada a $|u| \le 10$ N.
Con un carro de masa $M$ y una masa puntual $m$ en el extremo de una varilla de longitud $l$:

$$
\ddot x = \frac{u + m \sin\theta\,(l\dot\theta^2 - g\cos\theta)}{M + m\sin^2\theta}, \qquad
\ddot\theta = \frac{g\sin\theta - \ddot x\cos\theta}{l}
$$

Integramos con **Runge-Kutta 4** y un periodo de control de $\Delta t = 20$ ms (50 Hz), como haría un PLC o un microcontrolador.
El raíl mide $\pm 1.2$ m: si el carro llega al tope o el péndulo pasa de $\pm 0.6$ rad (≈ 34°) el ensayo **falla**.

Cerca de $\theta = 0$ la dinámica se puede **linealizar** ($\sin\theta \approx \theta$, $\cos\theta \approx 1$): $\dot s = A s + B u$. El LQR se diseña con
ese modelo lineal; el agente de RL **no** lo verá nunca.

In [ ]:
G = 9.81
DT = 0.02          # periodo de control [s] (50 Hz)
TH_FAIL = 0.6      # ángulo a partir del cual el ensayo se da por perdido [rad]


@dataclass(frozen=True)
class CartPole:
    M: float = 1.0      # masa del carro [kg]
    m: float = 0.3      # masa en la punta del péndulo [kg]
    l: float = 0.5      # longitud del péndulo [m]
    u_max: float = 10.0 # fuerza máxima del actuador [N]
    x_max: float = 1.2  # semirrecorrido del raíl [m]

    def with_mass_factor(self, k):
        """Misma planta con la masa del péndulo multiplicada por k (carga distinta)."""
        return replace(self, m=self.m * k)

    def f(self, s, u):
        """Dinámica no lineal: devuelve ds/dt."""
        x_d, th, w = s[1], s[2], s[3]
        S, C = np.sin(th), np.cos(th)
        xdd = (u + self.m * S * (self.l * w**2 - G * C)) / (self.M + self.m * S**2)
        thdd = (G * S - xdd * C) / self.l
        return np.array([x_d, xdd, w, thdd])

    def linear(self):
        """Linealización en la vertical superior: ds/dt = A s + B u."""
        M, m, l = self.M, self.m, self.l
        A = np.array([[0, 1, 0, 0],
                      [0, 0, -m * G / M, 0],
                      [0, 0, 0, 1],
                      [0, 0, G * (M + m) / (M * l), 0]], dtype=float)
        B = np.array([[0], [1 / M], [0], [-1 / (M * l)]], dtype=float)
        return A, B


def rk4(f, s, u, dt, substeps=4):
    h = dt / substeps
    for _ in range(substeps):
        k1 = f(s, u); k2 = f(s + 0.5 * h * k1, u); k3 = f(s + 0.5 * h * k2, u); k4 = f(s + h * k3, u)
        s = s + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return s


def wrap(th):
    """Ángulo en [-pi, pi)."""
    return (np.asarray(th) + np.pi) % (2 * np.pi) - np.pi


A, B = CartPole().linear()
eig = np.linalg.eigvals(A)
print('Autovalores de A (lazo abierto):', np.round(np.sort(eig.real), 3))
lam = eig.real.max()
print(f'Polo inestable: {lam:.2f} 1/s -> el ángulo se multiplica por e cada {1000 / lam:.0f} ms si nadie actúa')

### El entorno: la misma interfaz que en n16

Envolvemos la física en un **entorno** con la interfaz habitual de RL (`reset()`, `step(u)` → observación, recompensa, fin), pero la fuerza es
**continua en newtons**, así que los controladores clásicos pueden usar exactamente el mismo entorno. Añadimos lo que hace interesante una prueba
industrial: **empujones** (perturbaciones de fuerza), **ruido de medida** en los sensores y una **masa** del péndulo distinta de la nominal.

**Recompensa** (por paso, máximo 1): $\;r = 1 - (\theta/0.6)^2 - 0.5\,(x/1.2)^2 - 0.01\,(u/10)^2$, y $r=-10$ si el ensayo falla.
Es casi un coste cuadrático del LQR cambiado de signo: premia el péndulo vertical, el carro centrado y poco esfuerzo.

In [ ]:
class CartPoleEnv:
    """Carro-péndulo con empujones, ruido de medida y masa variable. step(u) recibe la fuerza en N."""

    def __init__(self, plant=CartPole(), max_steps=300, noise=0.0, pushes=None, random_pushes=False,
                 push_range=(5, 20), mass_range=None, th_fail=TH_FAIL, term_penalty=-10.0, reward_fn=None, seed=0):
        self.plant0 = self.plant = plant
        self.max_steps, self.noise, self.pushes = max_steps, noise, list(pushes or [])
        self.random_pushes, self.push_range, self.mass_range = random_pushes, push_range, mass_range
        self.th_fail, self.term_penalty, self.reward_fn = th_fail, term_penalty, reward_fn
        self.rng = np.random.default_rng(seed)

    def reset(self, state=None):
        if self.mass_range is not None:   # aleatorización de dominio (ejercicio 5)
            self.plant = self.plant0.with_mass_factor(self.rng.uniform(*self.mass_range))
        if state is None:                 # arranque aleatorio cerca de la vertical
            state = self.rng.uniform([-0.3, -0.2, -0.15, -0.2], [0.3, 0.2, 0.15, 0.2])
        self.s, self.k = np.array(state, float), 0
        self.ep_pushes = list(self.pushes)   # lista de (t_inicio [s], duración [s], fuerza [N])
        if self.random_pushes:
            t0 = self.rng.uniform(1.0, self.max_steps * DT - 1.0)
            self.ep_pushes.append((t0, 0.1, self.rng.choice([-1, 1]) * self.rng.uniform(*self.push_range)))
        return self.observe()

    def observe(self):
        """Lo que miden los sensores: estado + ruido (noise=1 -> 1 cm, 0.6°, y 5 cm/s, 0.05 rad/s en velocidades)."""
        if self.noise <= 0:
            return self.s.copy()
        return self.s + self.noise * np.array([0.01, 0.05, 0.01, 0.05]) * self.rng.standard_normal(4)

    def push_force(self, t):
        return sum(F for (t0, d, F) in self.ep_pushes if t0 <= t < t0 + d - 1e-9)

    def reward(self, s, u):
        if self.reward_fn is not None:
            return self.reward_fn(s, u)
        return 1.0 - (wrap(s[2]) / TH_FAIL) ** 2 - 0.5 * (s[0] / self.plant.x_max) ** 2 - 0.01 * (u / self.plant.u_max) ** 2

    def step(self, u):
        u = float(np.clip(u, -self.plant.u_max, self.plant.u_max))   # saturación del actuador
        d = self.push_force(self.k * DT)
        self.s = rk4(self.plant.f, self.s, u + d, DT)
        self.k += 1
        fell, rail = abs(wrap(self.s[2])) > self.th_fail, abs(self.s[0]) > self.plant.x_max
        term = bool(fell or rail)
        r = self.term_penalty if term else self.reward(self.s, u)
        return self.observe(), r, term, self.k >= self.max_steps, dict(push=d, u=u, fell=fell, rail=rail)


def rollout(env, ctrl, state=None, T=None):
    """Ejecuta un episodio con el controlador ctrl(obs) -> fuerza. Devuelve trayectoria y retorno."""
    if T is not None:
        env.max_steps = T
    o = env.reset(state)
    if hasattr(ctrl, 'reset'):
        ctrl.reset()
    S, U, P, ret, tc = [env.s.copy()], [], [], 0.0, 0.0
    term = False
    for _ in range(env.max_steps):
        t0 = time.perf_counter(); u = ctrl(o); tc += time.perf_counter() - t0
        o, r, term, trunc, info = env.step(u)
        S.append(env.s.copy()); U.append(info['u']); P.append(info['push']); ret += r
        if term or trunc:
            break
    return dict(s=np.array(S), u=np.array(U), push=np.array(P), ret=ret, failed=term,
                t=np.arange(len(S)) * DT, us_per_action=1e6 * tc / max(len(U), 1))

## 2. Verlo: animaciones del carro-péndulo

Cada panel muestra el carro, el péndulo (con una estela de la punta), la **fuerza del controlador** (flecha azul bajo el carro), los
**empujones** (flecha roja), el estado (**ESTABLE** / **RECUPERANDO** / **CAÍDO**) y un marco naranja si el carro llega al tope del raíl.
Debajo, la evolución del ángulo de todas las ejecuciones a la vez.

> Las animaciones usan `FuncAnimation` → reproductor HTML (`to_jshtml`), que funciona en Colab y en VS Code. Si tu entorno no las muestra,
> pon `ANIM_MODE = 'static'` (tira de fotogramas). Las principales se guardan también como GIF en `figures_cartpole/`.

In [ ]:
COL = dict(ink='#0f172a', cart='#334155', pole='#1e293b', bob='#2563eb', rail='#cbd5e1', push='#dc2626',
           force='#2563eb', ok='#16a34a', warn='#f59e0b', bad='#dc2626', grid='#e2e8f0', muted='#64748b')
CTRL_COLORS = {'LQR': '#2563eb', 'PID': '#7c3aed', 'DQN': '#ea580c', 'Aleatorio': '#94a3b8', 'Sin': '#94a3b8',
               'Supervisor': '#0d9488', 'Modelo': '#0891b2'}
EXTRA_COLORS = ['#fdba74', '#fb923c', '#ea580c', '#9a3412', '#0d9488', '#be185d']
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': '#cbd5e1',
                     'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': '#eef2f7',
                     'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'animation.embed_limit': 80,
                     'figure.dpi': 80})


def status(th, failed):
    if failed or abs(th) > TH_FAIL:
        return 'CAÍDO', COL['bad']
    if abs(th) < 0.05:
        return 'ESTABLE', COL['ok']
    return 'RECUPERANDO', COL['warn']


class CartPoleArtist:
    """Dibuja el carro-péndulo en un eje y lo actualiza fotograma a fotograma."""
    W, H, L_DRAW = 0.36, 0.18, 0.7

    def __init__(self, ax, plant=CartPole(), title=''):
        self.ax, self.p = ax, plant
        xm = plant.x_max
        ax.set_xlim(-xm - 0.35, xm + 0.35); ax.set_ylim(-0.40, 1.0); ax.set_aspect('equal'); ax.axis('off')
        ax.plot([-xm - 0.2, xm + 0.2], [-0.12, -0.12], color=COL['rail'], lw=3, solid_capstyle='round', zorder=1)
        for sgn in (-1, 1):   # topes del raíl
            ax.add_patch(Rectangle((sgn * (xm + self.W / 2) - 0.01, -0.14), 0.02, 0.22, color=COL['muted'], zorder=1))
        for xt in np.arange(-1.0, 1.01, 0.5):
            ax.plot([xt, xt], [-0.15, -0.18], color=COL['rail'], lw=1)
        self.cart = FancyBboxPatch((0, 0), self.W, self.H, boxstyle='round,pad=0.0,rounding_size=0.04', fc=COL['cart'], ec='none', zorder=3)
        ax.add_patch(self.cart)
        self.wheels = [Circle((0, 0), 0.035, fc=COL['ink'], zorder=4) for _ in range(2)]
        for w in self.wheels:
            ax.add_patch(w)
        self.pole, = ax.plot([], [], color=COL['pole'], lw=4, solid_capstyle='round', zorder=5)
        self.bob = Circle((0, 0), 0.055, fc=COL['bob'], ec='white', lw=1.5, zorder=6); ax.add_patch(self.bob)
        self.trail, = ax.plot([], [], color=COL['bob'], lw=1, alpha=0.25, zorder=2)
        ax.text(0.0, 1.0, title, transform=ax.transAxes, ha='left', va='top', fontsize=10, weight='bold', color=COL['ink'])
        self.info = ax.text(0.0, 0.90, '', transform=ax.transAxes, ha='left', va='top', fontsize=7.5, color=COL['muted'], family='monospace')
        self.stat = ax.text(1.0, 1.0, '', transform=ax.transAxes, ha='right', va='top', fontsize=9, weight='bold')
        self.push_txt = ax.text(0, 0.22, '', ha='center', fontsize=8.5, weight='bold', color=COL['push'], zorder=8)
        self.frame = Rectangle((0, 0), 1, 1, transform=ax.transAxes, fill=False, ec=COL['warn'], lw=4, visible=False, zorder=10)
        ax.add_patch(self.frame)
        self.arrows, self.hist = [], []

    def _arrow(self, x, y, dx, color, width):
        if abs(dx) < 1e-3:
            return
        a = FancyArrow(x, y, dx, 0, width=width, head_width=width * 3, head_length=min(0.08, abs(dx)),
                       length_includes_head=True, color=color, zorder=7)
        self.ax.add_patch(a); self.arrows.append(a)

    def update(self, s, t=0.0, u=0.0, push=0.0, failed=False):
        x, th, l = s[0], float(wrap(s[2])), self.L_DRAW   # l: longitud dibujada (escala visual)
        self.cart.set_x(x - self.W / 2); self.cart.set_y(-0.06)
        self.wheels[0].center = (x - self.W / 3, -0.075); self.wheels[1].center = (x + self.W / 3, -0.075)
        px, py = x + l * np.sin(th), 0.03 + l * np.cos(th)
        self.pole.set_data([x, px], [0.03, py]); self.bob.center = (px, py)
        self.hist.append((px, py)); h = np.array(self.hist[-25:]); self.trail.set_data(h[:, 0], h[:, 1])
        for a in self.arrows:
            a.remove()
        self.arrows = []
        self._arrow(x, -0.30, 0.04 * u, COL['force'], 0.018)                          # fuerza del controlador (0.4 m = 10 N)
        if push:
            self._arrow(x - np.sign(push) * 0.6, 0.03, np.sign(push) * 0.38, COL['push'], 0.03)   # empujón
            self.push_txt.set_position((x - np.sign(push) * 0.45, 0.12))
        self.push_txt.set_text(f'EMPUJÓN {push:+.0f} N' if push else '')
        st, c = status(th, failed)
        self.stat.set_text(st); self.stat.set_color(c)
        self.info.set_text(f't={t:4.1f}s  θ={np.degrees(th):+5.1f}°  x={x:+.2f}m  u={u:+5.1f}N')
        self.frame.set_visible(abs(x) > self.p.x_max - 0.05)


def _color_for(title, i, used):
    c = CTRL_COLORS.get(title.split()[0])
    if c is None or c in used:
        c = EXTRA_COLORS[i % len(EXTRA_COLORS)]
    used.add(c)
    return c


def animate_runs(runs, titles, every=3, cols=None, plot_theta=True, size=4.2, plant=CartPole()):
    """Un panel por ejecución (dicts de rollout) y, debajo, θ(t) de todas. Devuelve un FuncAnimation."""
    n = len(runs); cols = cols or min(n, 3); rows = int(np.ceil(n / cols))
    hr = [1.0] * rows + ([0.6] if plot_theta else [])
    fig = plt.figure(figsize=(size * cols, size * 0.5 * rows + (1.7 if plot_theta else 0)))
    gs = fig.add_gridspec(len(hr), cols, height_ratios=hr, hspace=0.12, wspace=0.05,
                          left=0.05, right=0.98, top=0.97, bottom=0.12 if plot_theta else 0.02)
    artists = [CartPoleArtist(fig.add_subplot(gs[i // cols, i % cols]), plant=plant, title=t) for i, t in enumerate(titles)]
    T = max(len(r['s']) for r in runs)
    if plot_theta:
        axt = fig.add_subplot(gs[-1, :]); axt.set_xlim(0, (T - 1) * DT); axt.set_ylim(-36, 36)
        axt.set_ylabel('θ [°]'); axt.set_xlabel('tiempo [s]'); axt.axhline(0, color=COL['grid'], lw=1)
        used = set()
        lines = [axt.plot([], [], lw=1.8, label=t, color=_color_for(t, i, used))[0] for i, t in enumerate(titles)]
        axt.legend(loc='upper right', fontsize=8, ncol=min(n, 4), frameon=False)
        cursor = axt.axvline(0, color=COL['muted'], lw=0.8, ls=':')
        pk = np.nonzero(runs[0]['push'])[0]
        if len(pk):
            axt.axvspan(pk[0] * DT, (pk[-1] + 1) * DT, color=COL['push'], alpha=0.10, lw=0)

    def upd(k):
        if k == 0:
            for a in artists:
                a.hist = []
        for a, r in zip(artists, runs):
            i = min(k, len(r['s']) - 1); j = min(i, len(r['u']) - 1)
            a.update(r['s'][i], t=i * DT, u=r['u'][j] if len(r['u']) else 0.0,
                     push=r['push'][i] if i < len(r['push']) else 0.0, failed=r['failed'] and i == len(r['s']) - 1)
        if plot_theta:
            for ln, r in zip(lines, runs):
                i = min(k, len(r['s']) - 1); ln.set_data(r['t'][:i + 1], np.degrees(wrap(r['s'][:i + 1, 2])))
            cursor.set_xdata([k * DT])
        return []

    frames = list(range(0, T, every))
    anim = animation.FuncAnimation(fig, upd, frames=frames, interval=1000 * DT * every, blit=False)
    anim.upd, anim.frame_list, anim.fig = upd, frames, fig
    plt.close(fig)
    return anim


def filmstrip(anim, n=4):
    """Alternativa sin JavaScript: n fotogramas de la animación en una sola figura."""
    ks = [anim.frame_list[int(i)] for i in np.linspace(0, len(anim.frame_list) - 1, n)]
    imgs = []
    for k in ks:
        anim.upd(k); anim.fig.canvas.draw(); imgs.append(np.asarray(anim.fig.canvas.buffer_rgba()))
    fig, axs = plt.subplots(n, 1, figsize=(9, 2.6 * n * imgs[0].shape[0] / imgs[0].shape[1] * 1.6))
    for ax, im, k in zip(np.atleast_1d(axs), imgs, ks):
        ax.imshow(im); ax.axis('off'); ax.set_title(f't = {k * DT:.1f} s', fontsize=9, loc='left')
    plt.tight_layout(); plt.show()


def show_anim(anim, gif=None, fps=17, as_gif=False):
    """Muestra la animación (según ANIM_MODE) y, si se pide, la guarda como GIF.
    as_gif=True la muestra como imagen GIF (más ligera; es lo que usan los laboratorios con widgets)."""
    if SAVE_GIFS and gif:
        anim.save(FIG_DIR / gif, writer=animation.PillowWriter(fps=fps), dpi=72)
        print(f'GIF guardado en {FIG_DIR / gif}')
    if as_gif and ANIM_MODE != 'none':
        tmp = FIG_DIR / '_lab_tmp.gif'
        anim.save(tmp, writer=animation.PillowWriter(fps=fps), dpi=72)
        from IPython.display import Image
        display(Image(data=tmp.read_bytes(), format='gif'))
    elif ANIM_MODE == 'jshtml':
        display(HTML(anim.to_jshtml(default_mode='once')))
    elif ANIM_MODE == 'static':
        filmstrip(anim)


def plot_runs(runs, labels, title=''):
    """θ, x y u frente al tiempo para varias ejecuciones."""
    fig, axs = plt.subplots(1, 3, figsize=(13, 2.8))
    used = set()
    for i, (r, lab) in enumerate(zip(runs, labels)):
        c = _color_for(lab, i, used)
        axs[0].plot(r['t'], np.degrees(wrap(r['s'][:, 2])), color=c, label=lab + (' ✗' if r['failed'] else ''))
        axs[1].plot(r['t'], r['s'][:, 0], color=c)
        axs[2].plot(r['t'][1:], r['u'], color=c, lw=1)
    for ax, yl in zip(axs, ['θ [°]', 'x [m]', 'u [N]']):
        ax.set_xlabel('tiempo [s]'); ax.set_ylabel(yl)
        pk = np.nonzero(runs[0]['push'])[0]
        if len(pk):
            ax.axvspan(pk[0] * DT, (pk[-1] + 1) * DT, color=COL['push'], alpha=0.10, lw=0)
    axs[1].axhline(CartPole().x_max, color=COL['warn'], ls='--', lw=1); axs[1].axhline(-CartPole().x_max, color=COL['warn'], ls='--', lw=1)
    axs[0].legend(fontsize=8, frameon=False); fig.suptitle(title, x=0.01, ha='left', weight='bold')
    plt.tight_layout(); plt.show()


class ZeroForce:
    name = 'Sin control'
    def __call__(self, o):
        return 0.0


# Sin control: el péndulo arranca a solo 3° de la vertical y cae (dejamos que llegue a la horizontal)
env_free = CartPoleEnv(max_steps=75, th_fail=np.pi / 2)
r_free = rollout(env_free, ZeroForce(), state=[0, 0, 0.05, 0])
show_anim(animate_runs([r_free], ['Sin control (u = 0)'], every=2, size=5.5, cols=1))

### ✍️ Ejercicio 1 (física)
1. Cambia `CartPole(m=..., l=...)` en la celda de la física: ¿cómo cambia el polo inestable $\lambda$ si el péndulo es el doble de largo? ¿Es más fácil o más difícil de equilibrar? Relaciónalo con equilibrar un palo largo o uno corto en la mano.
2. Con el valor de $\lambda$ que obtengas, estima cuánto tarda el ángulo en pasar de 3° a 30° sin control y compáralo con la animación.

## 3. Control clásico: PID y LQR

**PID (dos lazos sumados).** Un lazo PD sobre el ángulo (lo urgente) y otro PD sobre la posición del carro (que no se salga del raíl):
$u = K_{p\theta}\,\theta + K_{d\theta}\,\dot\theta + K_{px}\,x + K_{dx}\,\dot x$. Ojo al signo del lazo de posición: para llevar el carro hacia
la izquierda primero hay que empujar **a la derecha**, para inclinar el péndulo hacia donde queremos ir (sistema de **fase no mínima**).
No usamos término integral: aquí no hay error estacionario que corregir y el integrador solo añadiría oscilación.

**LQR.** Discretizamos el modelo lineal ($A_d, B_d$ con $\Delta t$ = 20 ms) y buscamos la ganancia $K$ que minimiza
$J = \sum_k s_k^\top Q s_k + u_k^\top R u_k$, resolviendo la ecuación de Riccati. El resultado es otra realimentación de estado $u = -Ks$,
pero con las cuatro ganancias **calculadas a la vez a partir del modelo**, en lugar de ajustadas a mano.

In [ ]:
def c2d(A, B, dt):
    """Discretización exacta con retenedor de orden cero."""
    n, m = B.shape
    Mx = np.zeros((n + m, n + m)); Mx[:n, :n] = A; Mx[:n, n:] = B
    E = expm(Mx * dt)
    return E[:n, :n], E[:n, n:]


Q_LQR = np.diag([10.0, 1.0, 100.0, 1.0])   # pesos sobre [x, x_dot, theta, theta_dot]
R_LQR = np.array([[0.1]])                  # peso sobre el esfuerzo u


def dlqr(Ad, Bd, Q=Q_LQR, R=R_LQR):
    P = solve_discrete_are(Ad, Bd, Q, R)
    return np.linalg.solve(R + Bd.T @ P @ Bd, Bd.T @ P @ Ad), P


def lqr_gain(plant=CartPole(), Q=Q_LQR, R=R_LQR):
    Ad, Bd = c2d(*plant.linear(), DT)
    return dlqr(Ad, Bd, Q, np.atleast_2d(R))[0]


class LQR:
    name = 'LQR'
    def __init__(self, K, u_max=10.0):
        self.K, self.u_max = np.atleast_2d(K), u_max
    def __call__(self, s):
        e = np.array([s[0], s[1], wrap(s[2]), s[3]])
        return float(np.clip(-(self.K @ e)[0], -self.u_max, self.u_max))


class CascadePID:
    name = 'PID'
    def __init__(self, kp_th=40.0, kd_th=6.0, kp_x=4.0, kd_x=6.0, u_max=10.0):
        self.g, self.u_max = (kp_th, kd_th, kp_x, kd_x), u_max
    def __call__(self, s):
        kp_th, kd_th, kp_x, kd_x = self.g
        u = kp_th * wrap(s[2]) + kd_th * s[3] + kp_x * s[0] + kd_x * s[1]
        return float(np.clip(u, -self.u_max, self.u_max))


K_LQR = lqr_gain()
Ad, Bd = c2d(*CartPole().linear(), DT)
print('Ganancia LQR  K =', np.round(K_LQR, 2), ' (u = -K s, s = [x, x_dot, theta, theta_dot])')
print('Ganancias PID    =', CascadePID().g, ' (equivale a K =', -np.array(CascadePID().g)[[2, 3, 0, 1]], ')')
print('|polos| en lazo cerrado LQR:', np.round(np.abs(np.linalg.eigvals(Ad - Bd @ K_LQR)), 3), '(todos < 1 -> estable)')
lqr, pid = LQR(K_LQR), CascadePID()

In [ ]:
# Mismo ensayo para los dos: arranque a 6°, empujón de +15 N durante 0.1 s en t = 1.5 s
scenario = dict(max_steps=250, pushes=[(1.5, 0.1, 15.0)])
runs = [rollout(CartPoleEnv(**scenario), c, state=[0, 0, 0.1, 0]) for c in (lqr, pid, ZeroForce())]
for r, n in zip(runs, ['LQR', 'PID', 'Sin control']):
    print(f'{n:12s} retorno = {r["ret"]:6.1f}  {"CAE" if r["failed"] else "aguanta"}')
plot_runs(runs[:2], ['LQR', 'PID'], 'LQR y PID ante un empujón de 15 N')
show_anim(animate_runs(runs, ['LQR', 'PID', 'Sin control']), gif='01_lqr_pid_empujon.gif')

### 🎛️ Laboratorio 1: ajusta el PID o el LQR

Mueve los *sliders* y mira cómo responde la planta. Con el **PID** ajustas las cuatro ganancias a mano; con el **LQR** solo eliges los pesos
$Q_\theta$, $Q_x$ y $R$ y la ganancia se recalcula. El controlador se diseña **siempre con el modelo nominal**, aunque cambies la masa real.
Marca *animar* para ver la animación (tarda unos segundos). Si no tienes widgets, edita los argumentos de `lab_clasico(...)`.

In [ ]:
def lab_clasico(controlador='LQR', q_theta=100.0, q_x=10.0, r=0.1, kp_th=40.0, kd_th=6.0, kp_x=4.0, kd_x=6.0,
                masa=1.0, empujon=15.0, ruido=0.0, animar=False):
    if controlador == 'LQR':
        ctrl = LQR(lqr_gain(CartPole(), Q=np.diag([q_x, 1.0, q_theta, 1.0]), R=[[r]]))
        info = f'K = {np.round(ctrl.K[0], 1)}'
    else:
        ctrl = CascadePID(kp_th, kd_th, kp_x, kd_x); info = f'ganancias = {ctrl.g}'
    env = CartPoleEnv(CartPole().with_mass_factor(masa), max_steps=300, noise=ruido, pushes=[(1.5, 0.1, empujon)], seed=1)
    res = rollout(env, ctrl, state=[0, 0, 0.1, 0])
    esfuerzo = np.sum(res['u'] ** 2) * DT
    print(f'{controlador}: {info} | {"❌ CAE" if res["failed"] else "✅ aguanta"} | retorno {res["ret"]:.1f} | esfuerzo Σu²Δt = {esfuerzo:.1f}')
    plot_runs([res], [controlador], f'{controlador} · masa ×{masa} · empujón {empujon:+.0f} N · ruido ×{ruido}')
    if animar:
        show_anim(animate_runs([res], [controlador], size=5.5, cols=1), as_gif=True)


if USE_WIDGETS:
    sl = lambda v, a, b, s, d: widgets.FloatSlider(value=v, min=a, max=b, step=s, description=d, continuous_update=False,
                                                   style={'description_width': '90px'}, layout=widgets.Layout(width='320px'))
    w = dict(controlador=widgets.ToggleButtons(options=['LQR', 'PID'], value='LQR'),
             q_theta=sl(100, 1, 500, 1, 'Q_θ (LQR)'), q_x=sl(10, 0.1, 100, 0.1, 'Q_x (LQR)'), r=sl(0.1, 0.01, 5, 0.01, 'R (LQR)'),
             kp_th=sl(40, 0, 100, 1, 'Kp_θ (PID)'), kd_th=sl(6, 0, 20, 0.5, 'Kd_θ (PID)'),
             kp_x=sl(4, -5, 20, 0.5, 'Kp_x (PID)'), kd_x=sl(6, -5, 20, 0.5, 'Kd_x (PID)'),
             masa=sl(1, 0.25, 5, 0.25, 'masa ×'), empujon=sl(15, -40, 40, 1, 'empujón [N]'), ruido=sl(0, 0, 8, 0.5, 'ruido ×'),
             animar=widgets.Checkbox(value=False, description='animar'))
    ui = widgets.VBox([w['controlador'],
                       widgets.HBox([widgets.VBox([w['q_theta'], w['q_x'], w['r']]),
                                     widgets.VBox([w['kp_th'], w['kd_th'], w['kp_x'], w['kd_x']]),
                                     widgets.VBox([w['masa'], w['empujon'], w['ruido'], w['animar']])])])
    display(ui, widgets.interactive_output(lab_clasico, w))
else:
    print('ipywidgets no disponible (o HEADLESS=1): versión estática. Cambia los argumentos y vuelve a ejecutar.')
    lab_clasico('PID', empujon=15.0)

### ✍️ Ejercicio 2 (control clásico)
1. Con el **PID**, busca unas ganancias que aguanten un empujón de **20 N** sin tocar los topes. Anota las ganancias, el retorno y el esfuerzo. ¿Qué pasa si pones $K_{px}$ y $K_{dx}$ con el signo cambiado (negativos)?
2. Con el **LQR**, multiplica $R$ por 10 y luego divídelo por 10. ¿Qué cambia en la fuerza $u(t)$ y en cuánto se mueve el carro? Explica con vuestras palabras qué representa $R$.
3. Busca la masa máxima (masa ×) con la que el LQR nominal todavía aguanta el empujón de 15 N.

## 4. El mismo problema como aprendizaje por refuerzo

| Elemento del MDP | En este notebook | Equivalente en control |
|---|---|---|
| Estado $s$ | $[x, \dot x, \theta, \dot\theta]$, normalizado a ≈ [-1, 1] | vector de estado |
| Acción $a$ | 7 fuerzas discretas: −10, −6.7, −3.3, 0, 3.3, 6.7, 10 N | entrada $u$ (cuantizada) |
| Recompensa $r$ | $1 - (\theta/0.6)^2 - 0.5(x/1.2)^2 - 0.01(u/10)^2$; −10 si falla | −coste por etapa |
| Episodio | 6 s (300 pasos), arranque aleatorio y **un empujón aleatorio** de 5–20 N | ensayo |
| Descuento $\gamma$ | 0.99 → horizonte efectivo ≈ $1/(1-\gamma)$ = 100 pasos = 2 s | horizonte del coste |

Usamos **DQN** (como en n16) con tres mejoras estándar: *replay buffer*, red objetivo actualizada suavemente ($\tau$ = 0.005) y **Double DQN**
(la red principal elige la acción del siguiente estado y la red objetivo la valora, para no sobreestimar Q). Cada 5000 pasos evaluamos la
política sin exploración en 8 episodios fijos y **guardamos la mejor**: el DQN es inestable y la última red no siempre es la mejor.

Primero, el agente **antes de aprender**: una red con pesos aleatorios.

In [ ]:
ACTIONS = np.linspace(-10, 10, 7)                  # fuerzas que puede elegir el agente [N]
OBS_SCALE = np.array([1.2, 2.0, TH_FAIL, 3.0])     # normalización del estado de entrada a la red


def norm_obs(o):
    return (np.array([o[0], o[1], wrap(o[2]), o[3]]) / OBS_SCALE).astype(np.float32)


class QNet(nn.Module):
    """Q(s, ·): 4 entradas -> 2 capas ocultas -> un valor Q por acción."""
    def __init__(self, n_actions=len(ACTIONS), hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_actions))
    def forward(self, x):
        return self.net(x)


class DQNController:
    """Política greedy de una QNet, con la misma interfaz que LQR y PID: ctrl(obs) -> fuerza."""
    name = 'DQN'
    def __init__(self, q, actions=ACTIONS):
        self.q, self.actions = q, actions
    @torch.no_grad()
    def __call__(self, o):
        return float(self.actions[self.q(torch.from_numpy(norm_obs(o))).argmax().item()])


torch.manual_seed(0)
q_random = QNet()
print(f'Parámetros de la QNet: {sum(p.numel() for p in q_random.parameters()):,}')
r_before = rollout(CartPoleEnv(max_steps=250, pushes=[(1.5, 0.1, 15.0)]), DQNController(q_random), state=[0, 0, 0.1, 0])
print(f'Agente sin entrenar: retorno {r_before["ret"]:.1f}, {"cae" if r_before["failed"] else "aguanta"} en t = {r_before["t"][-1]:.2f} s')
show_anim(animate_runs([r_before], ['DQN sin entrenar (antes)'], every=2, size=5.5, cols=1))

In [ ]:
def evaluate(ctrl, n=8, seed=999, T=500, **env_kw):
    """Retorno medio de un controlador en n episodios fijos de 10 s (arranque aleatorio + un empujón aleatorio)."""
    env = CartPoleEnv(max_steps=T, seed=seed, random_pushes=True, **env_kw)
    return float(np.mean([rollout(env, ctrl)['ret'] for _ in range(n)]))


def train_dqn(total_steps=TOTAL_STEPS, seed=0, gamma=0.99, lr=5e-4, batch=128, buf_size=50_000, start=1_000,
              eps_end=0.05, eps_frac=0.3, tau=0.005, hidden=128, actions=ACTIONS, env_kwargs=None,
              eval_every=5_000, snapshots=(), verbose=True):
    """DQN (Double DQN + replay + red objetivo suave). Devuelve la mejor red y el historial."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = CartPoleEnv(seed=seed, **(env_kwargs or dict(random_pushes=True)))
    q = QNet(len(actions), hidden); q_target = copy.deepcopy(q)
    opt = torch.optim.Adam(q.parameters(), lr=lr)
    # replay buffer circular en arrays de NumPy
    S = np.zeros((buf_size, 4), np.float32); A_ = np.zeros(buf_size, np.int64); R = np.zeros(buf_size, np.float32)
    S2 = np.zeros((buf_size, 4), np.float32); D = np.zeros(buf_size, np.float32); n = 0
    hist = dict(step=[], ret=[], length=[], eps=[], loss=[], eval_step=[], eval_ret=[])
    snaps, best = {}, (-np.inf, None)
    o, ep_ret, ep_len, losses, t0 = norm_obs(env.reset()), 0.0, 0, [], time.time()
    for t in range(total_steps):
        if t in snapshots:
            snaps[t] = copy.deepcopy(q.state_dict())
        eps = max(eps_end, 1 - (1 - eps_end) * t / (eps_frac * total_steps))      # ε-greedy con decaimiento lineal
        if random.random() < eps:
            a = random.randrange(len(actions))
        else:
            with torch.no_grad():
                a = q(torch.from_numpy(o)).argmax().item()
        o2, r, term, trunc, _ = env.step(actions[a]); o2 = norm_obs(o2)
        i = n % buf_size; S[i], A_[i], R[i], S2[i], D[i] = o, a, r, o2, float(term); n += 1
        o, ep_ret, ep_len = o2, ep_ret + r, ep_len + 1
        if term or trunc:            # 'trunc' (fin de tiempo) NO es un estado terminal: se sigue haciendo bootstrap
            hist['step'].append(t); hist['ret'].append(ep_ret); hist['length'].append(ep_len); hist['eps'].append(eps)
            o, ep_ret, ep_len = norm_obs(env.reset()), 0.0, 0
        if n >= start:
            idx = np.random.randint(0, min(n, buf_size), batch)
            s, a_b, r_b = torch.from_numpy(S[idx]), torch.from_numpy(A_[idx]), torch.from_numpy(R[idx])
            s2, d_b = torch.from_numpy(S2[idx]), torch.from_numpy(D[idx])
            with torch.no_grad():
                a2 = q(s2).argmax(1, keepdim=True)                                      # Double DQN: elige la red principal...
                y = r_b + gamma * (1 - d_b) * q_target(s2).gather(1, a2).squeeze(1)     # ...y valora la red objetivo
            loss = nn.functional.smooth_l1_loss(q(s).gather(1, a_b[:, None]).squeeze(1), y)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(q.parameters(), 10); opt.step()
            losses.append(loss.item())
            with torch.no_grad():
                for p, pt in zip(q.parameters(), q_target.parameters()):
                    pt.mul_(1 - tau).add_(tau * p)
        if (t + 1) % eval_every == 0:
            ev = evaluate(DQNController(q, actions))
            hist['eval_step'].append(t + 1); hist['eval_ret'].append(ev); hist['loss'].append(np.mean(losses[-eval_every:]) if losses else np.nan)
            if ev >= best[0]:
                best = (ev, copy.deepcopy(q.state_dict()))
            if verbose:
                print(f'{t + 1:7,d} pasos | ε={eps:.2f} | retorno entreno (10 últ.) {np.mean(hist["ret"][-10:]):6.1f} | '
                      f'evaluación {ev:6.1f} | {time.time() - t0:4.0f} s', flush=True)
    snaps[total_steps] = copy.deepcopy(q.state_dict())
    hist['wall_s'], hist['best_eval'] = time.time() - t0, best[0]
    q.load_state_dict(best[1])
    return q, hist, snaps

In [ ]:
# 🏋️ Entrenamiento (≈ 1.5–3 min en CPU con 80 000 pasos)
SNAPSHOTS = (0, TOTAL_STEPS // 8, TOTAL_STEPS // 4, TOTAL_STEPS // 2)
q_dqn, hist, snaps = train_dqn(TOTAL_STEPS, snapshots=SNAPSHOTS)
dqn = DQNController(q_dqn)
ref_lqr, ref_pid = evaluate(lqr), evaluate(pid)
print(f'\nTiempo de entrenamiento: {hist["wall_s"]:.0f} s | {len(hist["ret"])} episodios | {TOTAL_STEPS:,} transiciones')
print(f'Evaluación (8 episodios de 10 s, máx. 500): DQN mejor = {hist["best_eval"]:.1f} | LQR = {ref_lqr:.1f} | PID = {ref_pid:.1f}')

In [ ]:
# 📈 Curvas de aprendizaje
fig, axs = plt.subplots(1, 4, figsize=(15, 3))
st, ret = np.array(hist['step']), np.array(hist['ret'])
axs[0].plot(st, ret, '.', ms=2, color=CTRL_COLORS['DQN'], alpha=0.35, label='episodio de entrenamiento')
mov = pd.Series(ret).rolling(20, min_periods=1).mean()
axs[0].plot(st, mov, color=CTRL_COLORS['DQN'], lw=2, label='media móvil (20)')
axs[0].set_title('Retorno en entrenamiento (6 s, máx. 300)'); axs[0].legend(fontsize=7, frameon=False)
axs[1].plot(hist['eval_step'], hist['eval_ret'], 'o-', color=CTRL_COLORS['DQN'], label='DQN (sin exploración)')
axs[1].axhline(ref_lqr, color=CTRL_COLORS['LQR'], ls='--', label=f'LQR {ref_lqr:.0f}')
axs[1].axhline(ref_pid, color=CTRL_COLORS['PID'], ls=':', label=f'PID {ref_pid:.0f}')
axs[1].set_title('Evaluación (10 s, máx. 500)'); axs[1].legend(fontsize=7, frameon=False)
axs[2].plot(st, np.array(hist['length']) * DT, '.', ms=2, color=COL['muted']); axs[2].set_title('Duración del episodio [s]')
axs[3].plot(st, hist['eps'], color=COL['muted'], label='ε')
ax3b = axs[3].twinx(); ax3b.plot(hist['eval_step'], hist['loss'], color=COL['bad'], lw=1, label='pérdida Huber')
axs[3].set_title('Exploración ε y pérdida'); axs[3].legend(loc='upper center', fontsize=7, frameon=False); ax3b.legend(loc='upper right', fontsize=7, frameon=False)
for ax in axs:
    ax.set_xlabel('pasos de interacción')
plt.tight_layout(); plt.savefig(FIG_DIR / 'curvas_aprendizaje.png', dpi=110, bbox_inches='tight'); plt.show()

### Durante el entrenamiento: la misma prueba con redes de distintos momentos

Guardamos copias de la red a lo largo del entrenamiento. Ahora las ponemos **a la vez** ante el mismo arranque y el mismo empujón.

In [ ]:
test_case = dict(max_steps=300, pushes=[(2.0, 0.1, -15.0)])
snap_runs, snap_titles = [], []
for k in sorted(snaps)[:-1] + ['best']:
    qk = QNet()
    qk.load_state_dict(q_dqn.state_dict() if k == 'best' else snaps[k])
    snap_runs.append(rollout(CartPoleEnv(**test_case), DQNController(qk), state=[0.2, 0, -0.1, 0]))
    snap_titles.append('DQN final (mejor)' if k == 'best' else f'DQN {k // 1000}k pasos')
for r, t in zip(snap_runs, snap_titles):
    print(f'{t:20s} retorno {r["ret"]:6.1f}  {"cae en t=%.2f s" % r["t"][-1] if r["failed"] else "aguanta 6 s"}')
show_anim(animate_runs(snap_runs, snap_titles, cols=3, size=3.8), gif='02_dqn_durante_entrenamiento.gif')

### Después: DQN frente a LQR y PID en la misma prueba

In [ ]:
test_after = dict(max_steps=300, pushes=[(1.5, 0.1, 18.0)])
runs_after = [rollout(CartPoleEnv(**test_after), c, state=[-0.3, 0, 0.12, 0]) for c in (lqr, pid, dqn)]
for r, n in zip(runs_after, ['LQR', 'PID', 'DQN']):
    print(f'{n}: retorno {r["ret"]:6.1f} | {"CAE" if r["failed"] else "aguanta"} | Σu²Δt = {np.sum(r["u"] ** 2) * DT:6.1f} | '
          f'variación media de u = {np.mean(np.abs(np.diff(r["u"]))):.2f} N/paso')
plot_runs(runs_after, ['LQR', 'PID', 'DQN'], 'Después del entrenamiento: empujón de 18 N')
show_anim(animate_runs(runs_after, ['LQR', 'PID', 'DQN']), gif='03_dqn_vs_lqr_pid.gif')

## 5. ¿Qué ha aprendido el agente? Mapa de la política

Con el carro centrado y parado ($x = \dot x = 0$), dibujamos la fuerza que elige cada controlador para cada combinación de ángulo
$\theta$ y velocidad angular $\dot\theta$. La ley del LQR es un **plano** (lineal) que satura en ±10 N; el DQN solo puede elegir 7 fuerzas,
así que su mapa es una **escalera**. Si ha aprendido bien, las dos superficies deberían parecerse cerca del origen. A la derecha,
$V(s) = \max_a Q(s, a)$: lo que el agente cree que «vale» cada estado (el equivalente aprendido del coste $s^\top P s$ del LQR, cambiado de signo).

In [ ]:
th_g, w_g = np.linspace(-0.5, 0.5, 121), np.linspace(-2.5, 2.5, 121)
TH, W = np.meshgrid(th_g, w_g)
grid = np.stack([np.zeros_like(TH), np.zeros_like(TH), TH, W], -1).reshape(-1, 4)
u_lqr = np.clip(-(grid @ K_LQR.T)[:, 0], -10, 10).reshape(TH.shape)
with torch.no_grad():
    qv = q_dqn(torch.from_numpy((grid / OBS_SCALE).astype(np.float32))).numpy()
u_dqn, V = ACTIONS[qv.argmax(1)].reshape(TH.shape), qv.max(1).reshape(TH.shape)
_, P_lqr = dlqr(Ad, Bd)
V_lqr = -np.einsum('ij,jk,ik->i', grid, P_lqr, grid).reshape(TH.shape)

fig, axs = plt.subplots(1, 4, figsize=(16, 3.4))
ext = [np.degrees(th_g[0]), np.degrees(th_g[-1]), w_g[0], w_g[-1]]
for ax, Z, t in zip(axs[:2], [u_lqr, u_dqn], ['Fuerza del LQR [N]', 'Fuerza del DQN [N]']):
    im = ax.imshow(Z, origin='lower', extent=ext, aspect='auto', cmap='RdBu_r', vmin=-10, vmax=10); ax.set_title(t)
plt.colorbar(im, ax=axs[1])
agree = np.mean(np.sign(u_lqr) == np.sign(u_dqn))
im2 = axs[2].imshow(np.abs(u_lqr - u_dqn), origin='lower', extent=ext, aspect='auto', cmap='Greys', vmin=0, vmax=10)
axs[2].set_title(f'|diferencia|  (mismo signo en el {100 * agree:.0f} %)'); plt.colorbar(im2, ax=axs[2])
im3 = axs[3].imshow(V, origin='lower', extent=ext, aspect='auto', cmap='viridis'); axs[3].set_title('V(s) = max_a Q(s,a) del DQN')
axs[3].contour(np.degrees(TH), W, V_lqr, levels=8, colors='white', linewidths=0.6)
plt.colorbar(im3, ax=axs[3])
for ax in axs:
    ax.set_xlabel('θ [°]'); ax.set_ylabel('dθ/dt [rad/s]')
plt.tight_layout(); plt.savefig(FIG_DIR / 'mapa_politica.png', dpi=110, bbox_inches='tight'); plt.show()
print(f'El DQN y el LQR empujan en el mismo sentido en el {100 * agree:.0f} % de la rejilla (líneas blancas: curvas de nivel de -sᵀPs del LQR).')

### ✍️ Ejercicio 3 (interpretar)
1. ¿En qué zona del mapa coinciden más el DQN y el LQR, y en cuál menos? ¿Por qué el agente apenas ha visto estados con $|\theta| > 20°$ y $\dot\theta$ grande?
2. La forma de $V(s)$ (colores) y las curvas de nivel del LQR (líneas blancas): ¿se parecen? ¿Qué significa que $V$ sea más alto en la diagonal $\theta \cdot \dot\theta < 0$?

## 6. Comparativa RL frente a LQR: robustez

Un controlador industrial no solo tiene que funcionar en el caso nominal. Medimos:

- **Nominal:** 30 episodios de 10 s con arranque y empujón aleatorios (5–20 N, como en el entrenamiento).
- **Empujones:** fuerza creciente durante 0.1 s; ¿cuál es el mayor empujón que recupera cada uno?
- **Cambio de masa:** el péndulo pesa de 0.25 a 5 veces lo nominal (otra carga), con empujones pequeños (5–10 N). El LQR y el PID se diseñaron con la masa nominal y el DQN se entrenó con ella.
- **Ruido de medida:** los sensores añaden ruido gaussiano (×1 = 1 cm y 0.6°).

In [ ]:
CONTROLLERS = {'LQR': lqr, 'PID': pid, 'DQN': dqn}


def metrics(ctrl, n=30, seed=2024, **env_kw):
    env = CartPoleEnv(max_steps=500, seed=seed, random_pushes=True, **env_kw)
    rs = [rollout(env, ctrl) for _ in range(n)]
    return dict(retorno_medio=np.mean([r['ret'] for r in rs]),
                exito_pct=100 * np.mean([not r['failed'] for r in rs]),
                rms_theta_deg=np.mean([np.degrees(np.sqrt(np.mean(wrap(r['s'][:, 2]) ** 2))) for r in rs]),
                rms_x_m=np.mean([np.sqrt(np.mean(r['s'][:, 0] ** 2)) for r in rs]),
                esfuerzo=np.mean([np.sum(r['u'] ** 2) * DT for r in rs]),
                variacion_u=np.mean([np.mean(np.abs(np.diff(r['u']))) for r in rs]),
                us_por_accion=np.mean([r['us_per_action'] for r in rs]))


nominal = pd.DataFrame({k: metrics(c) for k, c in CONTROLLERS.items()}).T
nominal['datos_para_diseñar'] = ['modelo (ecuaciones)', 'ajuste manual', f'{TOTAL_STEPS:,} transiciones']
display(nominal.round(2))

In [ ]:
t0 = time.time()
# (a) empujones de fuerza creciente en t = 1 s, en los dos sentidos
forces = np.arange(0, 45.1, 2.5)
push_ok = {k: [np.mean([not rollout(CartPoleEnv(max_steps=250, pushes=[(1.0, 0.1, sgn * F)]), c, state=[0, 0, 0.05, 0])['failed']
                        for sgn in (-1, 1)]) for F in forces] for k, c in CONTROLLERS.items()}
# (b) masa del péndulo distinta de la nominal
mass_f = np.array([0.25, 0.5, 1, 1.5, 2, 3, 4, 5])
mass_ok = {k: [metrics(c, n=10, plant=CartPole().with_mass_factor(f), push_range=(5, 10))['exito_pct'] for f in mass_f] for k, c in CONTROLLERS.items()}
# (c) ruido de medida
noise_l = np.array([0, 1, 2, 4, 6, 8])
noise_m = {k: [metrics(c, n=10, noise=s) for s in noise_l] for k, c in CONTROLLERS.items()}
print(f'Barridos: {time.time() - t0:.0f} s')

fig, axs = plt.subplots(1, 4, figsize=(16, 3))
for k in CONTROLLERS:
    c = CTRL_COLORS[k]
    axs[0].plot(forces, 100 * np.array(push_ok[k]), 'o-', ms=3, color=c, label=k)
    axs[1].plot(mass_f, mass_ok[k], 'o-', ms=3, color=c, label=k)
    axs[2].plot(noise_l, [m['exito_pct'] for m in noise_m[k]], 'o-', ms=3, color=c, label=k)
    axs[3].plot(noise_l, [m['variacion_u'] for m in noise_m[k]], 'o-', ms=3, color=c, label=k)
axs[0].set(title='Recupera el empujón [%]', xlabel='fuerza [N] (0.1 s)')
axs[1].set(title='Éxito con otra masa [%]', xlabel='masa del péndulo / nominal'); axs[1].axvline(1, color=COL['muted'], ls=':')
axs[2].set(title='Éxito con ruido [%]', xlabel='ruido de medida (×)')
axs[3].set(title='Nerviosismo del actuador', xlabel='ruido de medida (×)', ylabel='media |Δu| [N/paso]')
axs[0].legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / 'robustez.png', dpi=110, bbox_inches='tight'); plt.show()


def max_ok(xs, oks, full=1.0):
    good = [x for x, o in zip(xs, oks) if o >= full]
    return max(good) if good else None

summary = pd.DataFrame({k: {'empujón máx. recuperado [N]': max_ok(forces, push_ok[k]),
                            'masa máx. con 100 % éxito [×]': max_ok(mass_f, mass_ok[k], 100),
                            'éxito con ruido ×4 [%]': noise_m[k][3]['exito_pct']} for k in CONTROLLERS}).T
display(summary)

In [ ]:
# 🔥 Prueba de estrés: péndulo el doble de pesado, ruido ×2 y dos empujones (+18 N y −18 N, casi el límite del entrenamiento)
stress = dict(plant=CartPole().with_mass_factor(2.0), noise=2.0, max_steps=400, pushes=[(1.5, 0.1, 18.0), (4.5, 0.1, -18.0)], seed=3)
runs_stress = [rollout(CartPoleEnv(**stress), c, state=[0, 0, 0.1, 0]) for c in CONTROLLERS.values()]
for r, n in zip(runs_stress, CONTROLLERS):
    print(f'{n}: {"CAE en t=%.2f s" % r["t"][-1] if r["failed"] else "aguanta"} | retorno {r["ret"]:.1f}')
show_anim(animate_runs(runs_stress, list(CONTROLLERS), plant=stress['plant']), gif='04_estres_rl_vs_lqr.gif')

### 🎛️ Laboratorio 2: la arena RL frente a control

Elige la masa real del péndulo, el empujón (fuerza e instante), el ruido y la inclinación inicial, y compara los controladores. Marca *animar* para ver la animación (GIF, tarda unos segundos).

In [ ]:
def arena(masa=1.0, empujon=15.0, t_empujon=1.5, ruido=0.0, theta0_deg=6.0, LQR_=True, PID_=True, DQN_=True, animar=False):
    sel = [k for k, on in zip(CONTROLLERS, (LQR_, PID_, DQN_)) if on]
    if not sel:
        print('Elige al menos un controlador'); return
    kw = dict(plant=CartPole().with_mass_factor(masa), noise=ruido, max_steps=300, pushes=[(t_empujon, 0.1, empujon)], seed=5)
    rs = [rollout(CartPoleEnv(**kw), CONTROLLERS[k], state=[0, 0, np.radians(theta0_deg), 0]) for k in sel]
    for r, k in zip(rs, sel):
        print(f'{k}: {"❌ CAE" if r["failed"] else "✅ aguanta"} | retorno {r["ret"]:6.1f} | Σu²Δt {np.sum(r["u"] ** 2) * DT:6.1f}')
    if animar:
        show_anim(animate_runs(rs, sel, plant=kw['plant']), as_gif=True)
    else:
        plot_runs(rs, sel)


if USE_WIDGETS:
    sl = lambda v, a, b, s, d: widgets.FloatSlider(value=v, min=a, max=b, step=s, description=d, continuous_update=False,
                                                   style={'description_width': '100px'})
    w2 = dict(masa=sl(1, 0.25, 5, 0.25, 'masa ×'), empujon=sl(15, -45, 45, 1, 'empujón [N]'), t_empujon=sl(1.5, 0.2, 5, 0.1, 't empujón [s]'),
              ruido=sl(0, 0, 8, 0.5, 'ruido ×'), theta0_deg=sl(6, -30, 30, 1, 'θ inicial [°]'),
              LQR_=widgets.Checkbox(value=True, description='LQR'), PID_=widgets.Checkbox(value=True, description='PID'),
              DQN_=widgets.Checkbox(value=True, description='DQN'), animar=widgets.Checkbox(value=False, description='animar'))
    display(widgets.HBox([widgets.VBox(list(w2.values())[:5]), widgets.VBox(list(w2.values())[5:])]),
            widgets.interactive_output(arena, w2))
else:
    print('ipywidgets no disponible (o HEADLESS=1): versión estática.')
    arena(masa=2.0, empujon=-15.0, animar=False)

## 7. Otra forma de aprender: aprender el **modelo** y luego hacer LQR

El DQN aprende la política **directamente** (RL *model-free*) y ha necesitado decenas de miles de transiciones. La alternativa clásica es la
**identificación de sistemas** (RL *model-based*, en el vocabulario de RL): con pocas transiciones $(s_k, u_k, s_{k+1})$ ajustamos por mínimos
cuadrados $s_{k+1} \approx \hat A_d s_k + \hat B_d u_k$ y diseñamos un LQR sobre ese modelo. Nadie le da las ecuaciones: solo datos.

In [ ]:
def identify_and_lqr(n_transitions, seed=0):
    """Recoge n transiciones con fuerzas aleatorias cerca de la vertical, ajusta (Ad, Bd) por mínimos cuadrados y diseña un LQR."""
    rng, env = np.random.default_rng(seed), CartPoleEnv(seed=seed)
    X, U, Xn = [], [], []
    while len(X) < n_transitions:
        env.reset()
        for _ in range(5):                     # episodios cortos para quedarnos cerca de la vertical
            s, u = env.s.copy(), rng.normal(0, 3)
            env.step(u); X.append(s); U.append([u]); Xn.append(env.s.copy())
    X, U, Xn = np.array(X[:n_transitions]), np.array(U[:n_transitions]), np.array(Xn[:n_transitions])
    theta, *_ = np.linalg.lstsq(np.c_[X, U], Xn, rcond=None)
    Ad_h, Bd_h = theta[:4].T, theta[4:].T
    return dlqr(Ad_h, Bd_h)[0]


rows = []
for n_tr in (10, 20, 50, 200):
    K_h = identify_and_lqr(n_tr)
    m = metrics(LQR(K_h), n=30)
    rows.append({'método': 'identificación + LQR', 'transiciones': n_tr, 'retorno_medio': m['retorno_medio'],
                 'exito_pct': m['exito_pct'], 'error_relativo_K_%': 100 * np.abs(K_h - K_LQR).max() / np.abs(K_LQR).max()})
rows.append({'método': 'DQN (model-free)', 'transiciones': TOTAL_STEPS, 'retorno_medio': nominal.loc['DQN', 'retorno_medio'],
             'exito_pct': nominal.loc['DQN', 'exito_pct'], 'error_relativo_K_%': np.nan})
rows.append({'método': 'LQR con el modelo exacto', 'transiciones': 0, 'retorno_medio': nominal.loc['LQR', 'retorno_medio'],
             'exito_pct': nominal.loc['LQR', 'exito_pct'], 'error_relativo_K_%': 0.0})
display(pd.DataFrame(rows).round(2))

## 8. Ejercicios para ir más allá

Cada ejercicio tiene una celda preparada que **no se ejecuta por defecto** (variable `RUN_... = False`). Cámbiala a `True`, ejecuta y
**anota vuestras cifras**: el cuestionario las pide.

- **✍️ Ejercicio 4 · La recompensa es el diseño.** Quita el término de posición $-0.5(x/1.2)^2$ de la recompensa y vuelve a entrenar
  (`RUN_REWARD = True`). ¿Qué hace ahora el carro? ¿Por qué? ¿Qué término del $Q$ del LQR juega el mismo papel?
- **✍️ Ejercicio 5 · Aleatorización de dominio (robustez).** Entrena con la masa del péndulo aleatoria entre 0.3× y 3× en cada episodio
  (`RUN_DR = True`) y repite el barrido de masa. ¿Mejora la robustez? ¿Cuesta algo en el caso nominal? Es la idea de *sim-to-real*.
- **✍️ Ejercicio 6 · Acciones.** Entrena con solo 3 acciones (−10, 0, 10 N, *bang-bang*) (`RUN_ACTIONS = True`). Compara la variación media de $u$
  (nerviosismo del actuador) con la versión de 7 acciones. ¿Qué implicaría para un motor real?
- **✍️ Ejercicio 7 · Supervisor de seguridad.** En planta no se deja a una red neuronal sin red de seguridad. Completa y prueba el
  `Supervisor`: usa el DQN, pero si el estado sale de una zona segura pasa el mando al LQR. Ajusta el umbral y pruébalo en la prueba de estrés.
- **Extra (avanzado):** arranca el péndulo **colgando** ($\theta = \pi$) e intenta que el DQN lo suba (*swing-up*). Pista: necesitas otra recompensa
  (por ejemplo, $\cos\theta$), episodios más largos y quitar el fallo por ángulo. El control clásico lo resuelve con un controlador de energía + LQR.

In [ ]:
RUN_REWARD = False   # ✍️ Ejercicio 4
if RUN_REWARD:
    no_x = lambda s, u: 1.0 - (wrap(s[2]) / TH_FAIL) ** 2 - 0.01 * (u / 10) ** 2
    q_nox, hist_nox, _ = train_dqn(TOTAL_STEPS, env_kwargs=dict(random_pushes=True, reward_fn=no_x))
    r_nox = rollout(CartPoleEnv(max_steps=500, pushes=[(1.5, 0.1, 15.0)]), DQNController(q_nox), state=[0, 0, 0.1, 0])
    show_anim(animate_runs([r_nox, rollout(CartPoleEnv(max_steps=500, pushes=[(1.5, 0.1, 15.0)]), dqn, state=[0, 0, 0.1, 0])],
                           ['DQN sin término x', 'DQN original']))

RUN_DR = False       # ✍️ Ejercicio 5
if RUN_DR:
    q_dr, hist_dr, _ = train_dqn(TOTAL_STEPS, env_kwargs=dict(random_pushes=True, mass_range=(0.3, 3.0)))
    dqn_dr = DQNController(q_dr)
    print('Éxito [%] frente a la masa:', dict(zip(mass_f, [metrics(dqn_dr, n=10, plant=CartPole().with_mass_factor(f), push_range=(5, 10))['exito_pct'] for f in mass_f])))
    print('Nominal:', metrics(dqn_dr))

RUN_ACTIONS = False  # ✍️ Ejercicio 6
if RUN_ACTIONS:
    bang = np.array([-10.0, 0.0, 10.0])
    q_bb, hist_bb, _ = train_dqn(TOTAL_STEPS, actions=bang)
    print('3 acciones:', metrics(DQNController(q_bb, bang)))
    print('7 acciones:', metrics(dqn))

In [ ]:
class Supervisor:
    """✍️ Ejercicio 7: DQN dentro de una zona segura; fuera de ella, LQR. Cambia el umbral y la zona."""
    name = 'Supervisor'

    def __init__(self, learned, fallback, theta_max_deg=15.0, x_max=0.9):
        self.learned, self.fallback = learned, fallback
        self.theta_max, self.x_max = np.radians(theta_max_deg), x_max
        self.n_fallback = 0

    def reset(self):
        self.n_fallback = 0

    def __call__(self, o):
        # TODO (alumnos): ¿qué otras variables deberían definir la zona segura? ¿Y si queremos histéresis?
        if abs(wrap(o[2])) > self.theta_max or abs(o[0]) > self.x_max:
            self.n_fallback += 1
            return self.fallback(o)
        return self.learned(o)


sup = Supervisor(dqn, lqr)
r_sup = rollout(CartPoleEnv(**stress), sup, state=[0, 0, 0.1, 0])
print(f'Supervisor en la prueba de estrés: {"CAE" if r_sup["failed"] else "aguanta"} | '
      f'pasos con LQR: {sup.n_fallback} de {len(r_sup["u"])}')

## 9. Conclusiones: ¿cuándo usar RL en control industrial?

- Con un **buen modelo** y un objetivo cuadrático, el **LQR** (o un MPC si hay restricciones) es difícil de batir: se calcula en milisegundos,
  no necesita datos y viene con garantías de estabilidad cerca del punto de trabajo. Bellman y Riccati son la misma ecuación en el caso lineal-cuadrático.
- El **RL** no necesitó las ecuaciones, pero sí **decenas de miles de ensayos** (en la planta real serían horas de pruebas y muchos péndulos
  en el suelo) y un **simulador**. Su comportamiento fuera de lo visto en el entrenamiento no está garantizado: hay que **medirlo** (robustez) y
  protegerlo (**supervisor**, límites, *safe RL*).
- El RL tiene sentido cuando el modelo es difícil de escribir (contactos, fricciones, procesos químicos, visión), la recompensa no es cuadrática
  (energía, calidad, tiempo de ciclo) o la tarea es muy no lineal (*swing-up*, manipulación). Ahí entran el aprendizaje por imitación,
  los modelos del mundo y el *sim-to-real* con **aleatorización de dominio**.
- Lo que más se usa hoy en planta es un camino intermedio: **aprender el modelo** (identificación, sección 7) y controlar con LQR/MPC, o usar RL
  para **ajustar** controladores clásicos.

📚 Referencias: Sutton y Barto, *Reinforcement Learning: An Introduction* (2.ª ed., 2018); Barto, Sutton y Anderson (1983), el carro-péndulo original;
Mnih et al. (2015), DQN; van Hasselt et al. (2016), Double DQN; Recht (2019), *A Tour of Reinforcement Learning: The View from Continuous Control*;
Åström y Murray, *Feedback Systems* (2.ª ed.).

👉 Ahora, el cuestionario **`n19_cartpole_rl_vs_control_quiz.md`**, con **vuestras** cifras.